# Predicting Payload Drops Early in a Session (DDS ch. 9, supervised)

Data-Driven Security ch. 9 splits machine learning into an **unsupervised** half (find structure nobody labelled; `campaigns.ipynb`) and a **supervised** half: learn a known label, and measure honestly how well that generalizes. This notebook is the supervised half. The question:

> From what a session shows at login, and in its first command or two, will it try to drop a payload?

That's a question a defender actually has: a session that is about to fetch malware is worth cutting off, recording in full or routing to a richer sandbox *before* the payload arrives.

**What makes it honest:**

- **No peeking at the future.** At horizon *k* the model sees the session up to its login (*k* = 0) or its *k*-th command, nothing after: not the session's length, not its closing events, not later commands.
- **No looking up the answer.** The label is nearly constant per IP, so an IP's history would turn the task into a lookup. The model gets no IP history and no campaign id (the clustering saw the commands and downloads).
- **Tested later and elsewhere.** The test set is the **last week**, which the models never see, evaluated once. Inside the training weeks, cross-validation folds are **grouped by ASN**, so a model is always scored on networks it didn't train on. The test week is also reported for **cold-start** sessions alone: networks never seen in training.

**Every run is versioned**, like the campaign runs: `pipeline.run()` hashes the config, the input files and the code into a run id, writes everything to `ml/outputs/runs/payload_drop/<run_id>/`, and **saves the trained model for each horizon** as an artifact, so the numbers below always belong to a model you can load and use.

In [ ]:
import json
import os
from pathlib import Path

import polars as pl
from IPython.display import Image

from honeypot.analysis.ml.payload.pipeline import EXPERIMENT, load_config, run
from honeypot.analysis.ml.runs import (
    experiment_dir,
    load_model,
    load_run,
    rebuild_index,
)

## Run (or reuse) the pipeline

`PayloadConfig()` holds every choice: horizons, folds, the model grid, the imbalance strategies, the target precision. Change any of it and you get a new run id; the old run stays. To use another config here, point `BEARTRAP_ML_PAYLOAD_CONFIG` at a JSON file of overrides (the same format as the CLI's `--config`). The first run takes about a quarter of an hour (most of it the cross-validation grid); afterwards this cell finds the finished run instantly.

In [ ]:
MODE = "offline"
CONFIG_PATH = os.environ.get("BEARTRAP_ML_PAYLOAD_CONFIG")
config = load_config(Path(CONFIG_PATH) if CONFIG_PATH else None, seed=None)

result = run(MODE, config=config)
loaded = load_run(result.path)
metrics, manifest = loaded["metrics"], loaded["manifest"]
charts = result.path / "charts"


def table(name: str) -> pl.DataFrame:
    return pl.read_parquet(result.path / "tables" / f"{name}.parquet")


pl.Config.set_tbl_rows(30)
pl.Config.set_fmt_str_lengths(100)
print(f"run {result.run_id} ({'reused' if result.skipped else 'new'}): {result.path}")
print(
    f"code {manifest['code_sha256'][:12]}, git {manifest['git_sha']} "
    f"(dirty: {manifest['git_dirty']})"
)
pl.DataFrame(
    [
        {
            "input": name,
            "file": item["path"].rsplit("/", 1)[-1],
            "sha256": item["sha256"][:12],
        }
        for name, item in manifest["inputs"].items()
    ]
)

## The label

A session with a successful login is **positive** when it tries to get a file onto the box: a fetch tool pointed at a network location (`wget` / `curl` / `tftp` / `ftpget` plus a URL or IP), `scp` from a remote host, a binary written out with `echo` (at least 32 hex-escaped bytes in one command), or a Cowrie transfer event (a download, failed download, or SFTP/SCP upload).

Two look-alikes are deliberately **not** positives: `/bin/busybox wget; /bin/busybox STC` only checks that `wget` exists, and `echo -e "\x6F\x6B"` prints "ok" as a liveness check. Both are common in Mirai-family playbooks.

In [ ]:
print(
    f"{metrics['data']['sessions']:,} sessions with a login, "
    f"{metrics['data']['drops']:,} drops "
    f"({metrics['data']['drops'] / metrics['data']['sessions']:.1%})"
)
table("label_summary")

## Horizons, populations and the split

At horizon *k* the population is the sessions with at least *k* commands that haven't dropped yet. A session that already dropped is decided, and one that ended before *k* commands is left out, otherwise "fewer than *k* commands" would give away "ended early, so negative". The test week starts at midnight UTC, `test_days` days before the end of the snapshot. *Cold start* = test-week sessions from ASN groups that never appear in the training weeks.

In [ ]:
print("test week from", metrics["data"]["test_start"])
table("population")

## Imbalance

About 3% of sessions drop a payload, so a model that always says "no" is 97% accurate and useless. Everything here is scored by **PR-AUC** (average precision), which only depends on how well the positives are ranked. A random ranking scores the base rate. Alongside it are ROC-AUC, the **Brier score** (squared error of the probabilities, i.e. calibration) and **recall at 80% precision**: the share of drops caught while 4 in 5 alerts are right.

Four ways to train with rare positives, each tried on every model's best parameters:

| strategy | what it does |
|---|---|
| none | train on the data as is |
| class_weight | weight each class inversely to its frequency |
| undersample | drop random negatives down to 1:1 |
| smote | synthesize new positives between neighbouring positives, up to 1:1 |

Resampling happens inside the training folds only (imbalanced-learn's `Pipeline` resamples in `fit`, never in `predict`), so held-out folds keep the real 3%. Resampling also teaches a model that positives are common: its *ranking* may be fine, but its probabilities run high, which the Brier score shows.

In [ ]:
Image(charts / "imbalance_strategies.png")

## Model comparison

DDS ch. 9's toolbox, simplest first: a constant (the majority baseline), a one-rule decision stump (1R), Bernoulli naive Bayes, logistic regression (L2, and L1 for sparse weights), a shallow decision tree, a linear SVM, *k*-nearest neighbours, a random forest and gradient boosting. Every candidate is scored on the same five ASN-grouped, stratified folds of the training weeks. Dots show the mean, bars one fold standard deviation; gray are the baselines, blue the chosen candidate.

In [ ]:
Image(charts / "model_comparison.png")

In [ ]:
cv = table("cv_summary")
cv.sort(["horizon", "pr_auc_mean"], descending=[False, True]).group_by(
    "horizon", maintain_order=True
).head(6).select(
    "horizon",
    "model",
    "params",
    "strategy",
    "pr_auc_mean",
    "pr_auc_sd",
    "roc_auc_mean",
    "brier_mean",
    "recall_at_precision_mean",
)

Note the **one-rule** baseline at *k* = 0: on all training weeks the best single split is a word from AS names (`limited`, in the 2026-09-16 snapshot), behind which almost every session drops. It belongs to a few networks, so on held-out networks the rule scores the base rate. That's memorizing a network, not learning behaviour, and it's exactly what the ASN-grouped folds are there to expose.

**Selection** (one-standard-error rule, on paired differences): every candidate is scored on the same folds, and how many positives a fold holds moves all of them together, so candidates are compared fold by fold. A candidate counts as tied with the best when its mean fold-by-fold shortfall is within one standard error of that shortfall. Among the tied, the **simplest** model wins. The folds can't tell them apart, and a simpler model is cheaper, more readable and less likely to have fit noise. The decision threshold comes from the chosen candidate's out-of-fold predictions on the training weeks: the one with the best F1, plus the lowest one reaching 80% precision.

In [ ]:
table("chosen").select(
    "horizon",
    "model",
    "params",
    "strategy",
    "pr_auc_mean",
    "tied_candidates",
    "best_candidate",
    "threshold_f1",
    "threshold_precision",
)

## What carries the signal

**Block ablation**: the chosen model is cross-validated again with one feature block left out at a time; the drop in PR-AUC is how much the model leans on that block (blocks that overlap can stand in for each other, so small changes don't mean useless). For the readable models, their own weights show *which* tokens matter: coefficients, naive-Bayes log-probability ratios or tree importances.

In [ ]:
Image(charts / "ablation.png")

In [ ]:
Image(charts / "feature_weights.png")

## The test week, once

The chosen model per horizon, refit on all training weeks, scores the test week: every session, and the cold-start subset. Intervals are 95% bootstrap intervals that resample **whole ASN groups**. A botnet's thousand near-identical sessions are one draw, not a thousand.

In [ ]:
table("test_metrics").select(
    "horizon",
    "view",
    "sessions",
    "positives",
    "cv_pr_auc",
    "pr_auc",
    "pr_auc_low",
    "pr_auc_high",
    "roc_auc",
    "brier",
    "recall_at_precision",
    "threshold_f1_precision",
    "threshold_f1_recall",
)

In [ ]:
Image(charts / "horizon_curve.png")

In [ ]:
Image(charts / "pr_curves.png")

**Calibration**: of the sessions the model gives about 30%, do about 30% drop? Points on the diagonal are calibrated. Models trained with resampling, or naive Bayes (which assumes features are independent and so double-counts correlated tokens), typically sit below it: right ranking, inflated probabilities. The threshold, not the raw probability, is what to act on.

In [ ]:
Image(charts / "calibration.png")

## Which campaigns does it miss?

Test-week sessions per campaign from the unsupervised half (the campaigns run on the same snapshot, recorded in this run's manifest), at the max-F1 threshold. Campaigns are only used here, after the fact. "unclustered" are IPs the campaign run never saw.

In [ ]:
Image(charts / "errors_by_campaign.png")

In [ ]:
table("errors_by_campaign").filter(pl.col("horizon") == 0).head(10)

## Using a saved model

Each run saves one fitted pipeline per horizon (encoder, resampler, classifier) as `models/horizon_<k>.joblib`, with metadata beside it. `load_model` checks the file's sha256 against the run's manifest before unpickling, so only a model this pipeline wrote, unchanged since, is ever loaded. It takes `horizon_rows()` output directly:

In [ ]:
model, meta = load_model(result.path, "horizon_0")
shown = {k: v for k, v in meta.items() if k not in ("feature_names", "packages")}
print(json.dumps(shown, indent=1))
print(len(meta["feature_names"]), "features")

## Run history

Every finished payload run, from `ml/outputs/runs/payload_drop/index.parquet` (rebuilt from the run directories after every run).

In [ ]:
index = rebuild_index(experiment_dir(EXPERIMENT))
columns = [
    c
    for c in index.columns
    if c in ("run_id", "finished_at", "input.snapshot")
    or (
        c.startswith("metrics.horizon_")
        and c.endswith(("chosen.model", "test.all.pr_auc"))
    )
]
index.select(columns)

## Caveats

- **The label is an attempt, not an infection.** Most drops are Mirai-family telnet loaders; the honeypot fakes the box they land on.
- **One deployment, one month.** Two sensors, 2026-08-17 to 2026-09-16. The test week shows the models carry over one week and to unseen networks; a new botnet with its own credentials and client is exactly what they can't have seen.
- **Near-duplicates.** Grouped folds and the group bootstrap limit, but don't remove, the optimism from botnets that send thousands of identical sessions.
- **Probabilities vs. ranking.** Selection is on PR-AUC (ranking). Read the chosen models' scores through their thresholds; the calibration plot shows how far the raw probabilities can be trusted.
- **One accidental look at the test week.** While sizing the model grid, a timing script fit default-parameter models on the training weeks and scored them on the test week, before any selection code existed. No decision used those numbers: the grid and the selection rule were fixed in the spec beforehand, and selection only ever sees cross-validation on the training weeks.